# Warm-starting a batched differentiable solve (pounce#78)

`JaxProblem.batched_solve` (from pounce#76) packs `B` parametric
subproblems into one stacked block-diagonal NLP so they share a
barrier homotopy and a symbolic factorisation — one Rust crossing
per batch. `JaxProblem.solve_with_warm` (from pounce#74) lets a
*single*-sample solve seed the IPM from a previous step's primal +
dual state, which cuts iteration count when the parameter drifts
only slightly between calls (the typical shape of a constrained-
projection layer inside a training loop).

`batched_solve_with_warm` is the intersection: the *batched* solve
with per-block warm vectors threaded in/out, so one stacked Rust
crossing serves the whole batch and each block reuses its previous
step's duals. This notebook shows the API and verifies (a) the
iteration count drops when warm-started, and (b) the gradient
matches `batched_solve` end-to-end.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from pounce.jax import JaxProblem

jax.config.update("jax_enable_x64", True)

## 1. The problem

Per-sample constrained projection: project a noisy prediction $\hat p \in \mathbb R^n$
onto an affine subspace inside box bounds.

$$
\min_x\;\tfrac12\|x-p\|^2 \quad \text{s.t.}\quad Ax = b,\;\; -10\le x\le 10.
$$

The constraint matrix $A,b$ is fixed across the batch — only the
target $p$ varies per block, which is exactly the shape
`batched_solve` was designed for.

In [2]:
N, M, B = 16, 4, 32
rng = np.random.default_rng(0)
A = jnp.asarray(rng.standard_normal((M, N)))
b = jnp.asarray(rng.standard_normal((M,)))

def f(x, p):
    return 0.5 * jnp.sum((x - p) ** 2)

def g(x, p):  # noqa: ARG001
    return A @ x - b

jp = JaxProblem(
    f=f, g=g, n=N, m=M, p_example=jnp.zeros(N),
    lb=jnp.full(N, -10.0), ub=jnp.full(N, 10.0),
    cl=jnp.zeros(M), cu=jnp.zeros(M),
    options={"tol": 1e-10, "print_level": 0, "sb": "yes"},
    factor_reuse=True,
)

## 2. Simulate a training loop

At each step the parameter batch $P$ drifts slightly (a small Gaussian
perturbation — stand-in for an outer-loop gradient update on a NN that
produces $\hat p$). We run two versions of the loop and compare
iteration counts:

* **Cold**: a fresh stacked solve every step from a zero seed.
* **Warm**: thread the previous step's converged $(\lambda, z_L, z_U)$ in.

To pull `iter_count` directly out of the IPM `info` dict we call the
underlying stacked `Problem` once per step — same machinery the
`_host_batched_solve` / `_host_batched_solve_warm` host hooks use,
just with the `info` kept visible.

In [3]:
STEPS = 6
STEP_NOISE = 0.05

def run_loop(warm: bool):
    """Mini training-loop simulation. Returns the list of IPM
    iteration counts at each step. Calls the cached stacked Problem
    directly so we can read ``info['iter_count']`` per step.
    """
    # Grab (or build) the cached stacked Problem on *this* thread.
    # Going through ``jp.batched_solve`` would route via a JAX
    # ``pure_callback`` that runs the host call on a different thread,
    # so the TLS cache wouldn't be visible here.
    if warm:
        obj, prob = jp._thread_stacked_problem_warm(B)
    else:
        obj, prob = jp._thread_stacked_problem(B)

    iters = []
    P_t = jnp.asarray(P)
    X0_flat = np.zeros(B * N)
    lam_prev = np.zeros(B * M)
    zL_prev = np.zeros(B * N)
    zU_prev = np.zeros(B * N)
    for t in range(STEPS):
        obj._P = P_t                              # update parameters
        if warm and t > 0:
            _, info = prob.solve(
                x0=X0_flat, lagrange=lam_prev,
                zl=zL_prev, zu=zU_prev,
            )
        else:
            _, info = prob.solve(x0=X0_flat)
        iters.append(int(info["iter_count"]))
        lam_prev = np.asarray(info["mult_g"], dtype=np.float64)
        zL_prev = np.asarray(info["mult_x_L"], dtype=np.float64)
        zU_prev = np.asarray(info["mult_x_U"], dtype=np.float64)
        P_t = P_t + STEP_NOISE * jnp.asarray(rng.standard_normal((B, N)))
    return iters

P = rng.standard_normal((B, N))
iters_cold = run_loop(warm=False)
iters_warm = run_loop(warm=True)
print(f"cold  iters per step: {iters_cold}")
print(f"warm  iters per step: {iters_warm}")
print(f"warm cumulative speedup: "
      f"{sum(iters_cold) / max(sum(iters_warm), 1):.2f}× fewer IPM iters")

cold  iters per step: [6, 6, 6, 6, 6, 6]
warm  iters per step: [5, 5, 5, 5, 5, 5]
warm cumulative speedup: 1.20× fewer IPM iters


## 3. The public surface

Above we called the cached stacked `Problem` directly so we could
read iteration counts. In real use you call `batched_solve_with_warm`
directly — the API mirrors `solve_with_warm`'s single-sample shape:

```python
x_batch, (lam_b, zL_b, zU_b) = jp.batched_solve_with_warm(
    p_batch, x0, warm_start=(lam_prev, zL_prev, zU_prev),
)
```

Pass `warm_start=None` (or omit) for the first call when no prior
state is available; the call is then equivalent to `batched_solve`
modulo the `warm_start_init_point="yes"` flag (which still helps
even from a zero seed because it relaxes the IPM's bound-push
shifts).

In [4]:
# One full step of the public API.
x_b, (lam_b, zL_b, zU_b) = jp.batched_solve_with_warm(jnp.asarray(P), jnp.zeros((B, N)))
print(f"x shape: {x_b.shape}   lam shape: {lam_b.shape}   zL shape: {zL_b.shape}")

# Next step: thread the just-converged duals back in.
P_next = jnp.asarray(P + STEP_NOISE * rng.standard_normal((B, N)))
x_b2, (lam_b2, zL_b2, zU_b2) = jp.batched_solve_with_warm(
    P_next, jnp.zeros((B, N)), warm_start=(lam_b, zL_b, zU_b),
)
print(f"step-2 x range: [{float(x_b2.min()):.3f}, {float(x_b2.max()):.3f}]")

x shape: (32, 16)   lam shape: (32, 4)   zL shape: (32, 16)
step-2 x range: [-3.297, 2.591]


## 4. Differentiability

`batched_solve_with_warm` is wired with `jax.custom_vjp`. The
backward treats the warm vectors and `x0` as stop-gradient (same
convention `solve_with_warm` uses) and computes
$\partial x^* / \partial p$ through the held stacked LDLᵀ factor
(`factor_reuse=True`). The result must match the gradient through
plain `batched_solve` to IPM tolerance — same KKT system, same
block-diagonal coupling.

In [5]:
P_j = jnp.asarray(P)
X0_j = jnp.zeros((B, N))

def loss_warm(P_):
    x, _ = jp.batched_solve_with_warm(P_, X0_j)
    return 0.5 * jnp.sum(x ** 2)

def loss_cold(P_):
    x = jp.batched_solve(P_, X0_j)
    return 0.5 * jnp.sum(x ** 2)

g_warm = jax.grad(loss_warm)(P_j)
g_cold = jax.grad(loss_cold)(P_j)
max_err = float(jnp.max(jnp.abs(g_warm - g_cold)))
print(f"max |grad warm - grad cold| = {max_err:.2e}")

max |grad warm - grad cold| = 4.44e-16


## When to reach for it

* **Inner training loop with a stacked NLP per step**: the parameter
  drifts only slightly between iterations, and you already use
  `batched_solve` to compress B subproblems into one Rust crossing.
  Threading warm duals through cuts the IPM iteration count per step,
  on top of the per-step amortisation `batched_solve` already gives.
* **First call**: pass `warm_start=None` — the API falls back to a
  cold start (zero warm vectors), so the calling code doesn't need a
  separate branch for the first step.
* **Not the right tool when** the batch is one-shot (no temporal
  coherence — the warm vectors carry no signal), per-block
  convergence diverges sharply (warm-starting a slow block from a
  fast block's seed can pessimise), or the per-block size is large
  enough that even one factorisation hurts — in that last case the
  open issue is reducing factor cost, not warm-starting.